# Claude and Giorgio's correction: data-quality audit of the June 2026 dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giorgio-Colombo/Data-Analysis-Portfolio/blob/main/clash-royale-2026-meta-econometrics/claude_correction_data_quality_audit.ipynb)

**What this is.** A review of the original analysis ([`clash_royale_2026_meta_econometrics.ipynb`](clash_royale_2026_meta_econometrics.ipynb)), written with the help of Claude Code. The original notebook is left unchanged so the two can be compared.

**Why it exists.** The original deck ranking contained results that cannot be real: decks with an **85-92% win rate** over hundreds of games, and some of the most-played decks at **10-21%**. Matchmaking keeps win rates close to 50%, so these numbers point to a data problem, not to deck strength. This notebook finds where they come from.

**Scope.** One file of the dataset (`2026-06-21_0.parquet`, 10 million matches), downloaded automatically. The dataset has many files, so the findings should be re-checked on the others before being generalised.

In [1]:
import json
import numpy as np
import pandas as pd
import kagglehub

HANDLE = "jackmangione/clash-royale-matchups-june2026"
FILE = "2026-06-21_0.parquet"

# Download only the one file we need, plus the card-name mapping
path = kagglehub.dataset_download(HANDLE, path=FILE)
with open(kagglehub.dataset_download(HANDLE, path="cardToID.json")) as f:
    id_to_card = {int(v): k for k, v in json.load(f).items()}

df = pd.read_parquet(path)
print(f"{len(df):,} rows")

10,000,000 rows


## 1. What is in the file

In [2]:
print(df["game_mode"].value_counts().to_string())
print("\nTime range:", df["time"].min(), "->", df["time"].max())

lvl_w = [f"winner_card_{i}_level" for i in range(8)]
lvl_l = [f"loser_card_{i}_level" for i in range(8)]
print("\nCard level (all cards):")
print(pd.concat([df[lvl_w].stack(), df[lvl_l].stack()]).describe().round(2).to_string())

game_mode
Ladder                9960921
Ranked1v1_NewArena      39079

Time range: 2026-05-22 16:16:15-04:00 -> 2026-06-04 02:57:08-04:00

Card level (all cards):


count    1.600000e+08
mean     1.337000e+01
std      2.600000e+00
min      1.000000e+00
25%      1.200000e+01
50%      1.400000e+01
75%      1.500000e+01
max      1.600000e+01


Almost all rows are `Ladder` matches, as the original analysis assumed. Note that the file is named `2026-06-21`, but its matches run from 22 May to 4 June: the file name is not the match date.

Next, the same preparation as the original notebook: keep ladder matches, encode each 8-card deck as one integer, and compute each deck's win rate.

In [3]:
df = df[df["game_mode"] == "Ladder"].reset_index(drop=True)

wc = [f"winner_card_{i}" for i in range(8)]
lc = [f"loser_card_{i}" for i in range(8)]

def encode(M):
    S = np.ascontiguousarray(np.sort(M, axis=1).astype(np.uint8))
    return S.view(np.uint64).ravel()

def names(code):
    ids = np.array([code], dtype=np.uint64).view(np.uint8)
    return " | ".join(id_to_card.get(int(i), str(int(i))) for i in ids)

df["wd"], df["ld"] = encode(df[wc].to_numpy()), encode(df[lc].to_numpy())
df["wlv"], df["llv"] = df[lvl_w].to_numpy().mean(1), df[lvl_l].to_numpy().mean(1)

def deck_table(d, min_games=300):
    t = pd.concat([d["wd"].value_counts().rename("w"),
                   d["ld"].value_counts().rename("l")], axis=1).fillna(0)
    t["n"] = t["w"] + t["l"]
    t["wr"] = t["w"] / t["n"]
    return t[t["n"] >= min_games]

def summarise(d, label):
    b = deck_table(d)
    q = b["wr"].quantile([.01, .10, .50, .90, .99]).round(3).tolist()
    print(f"{label}")
    print(f"  matches {len(d):,} | decks with >=300 games: {len(b):,}")
    print(f"  win-rate quantiles (1/10/50/90/99%): {q} | max {b['wr'].max():.3f} | min {b['wr'].min():.3f}")
    print(f"  match-weighted mean win rate: {(b['wr'] * b['n']).sum() / b['n'].sum():.3f}  (should be close to 0.5)")

summarise(df, "ALL ladder matches")

ALL ladder matches
  matches 9,960,921 | decks with >=300 games: 2,277
  win-rate quantiles (1/10/50/90/99%): [0.071, 0.148, 0.269, 0.634, 0.861] | max 0.932 | min 0.020
  match-weighted mean win rate: 0.329  (should be close to 0.5)


## 2. The anomaly, reproduced
The deck table below is the one the original notebook ranked as "strategically robust".

In [4]:
b = deck_table(df)
pd.DataFrame({
    "games": b["n"].astype(int),
    "win_rate": b["wr"].round(3),
    "cards": [names(c) for c in b.index],
}).sort_values("win_rate", ascending=False).head(8).reset_index(drop=True)

,games,win_rate,cards
0,409,0.932,knight | archers | goblins | giant | musketeer...
1,1030,0.926,knight | archers | goblins | giant | musketeer...
2,345,0.925,knight | archers | goblins | giant | minions |...
3,529,0.924,knight | goblins | giant | minions | musketeer...
4,699,0.924,knight | archers | goblins | giant | minions |...
5,307,0.919,knight | archers | goblins | giant | musketeer...
6,306,0.918,knight | archers | giant | mini p.e.k.k.a | sp...
7,647,0.912,knight | archers | giant | minions | mini p.e....


Win rates of 91-93% sustained over hundreds of games and hundreds of different players are not plausible under matchmaking.

## 3. Cause A: matches between beginners
Look at who plays the top decks.

In [5]:
top = b.sort_values("wr", ascending=False).index[:3]
ids_evo_hero = [i for i, v in id_to_card.items() if v.startswith(("evo", "hero"))]
share_evo = lambda M: np.isin(M, ids_evo_hero).any(1).mean()

rows = []
for d in top:
    won = df[df["wd"] == d]
    rows.append({
        "deck": names(d)[:60] + "...",
        "avg card level (winner)": round(won["wlv"].mean(), 2),
        "avg card level (opponent)": round(won["llv"].mean(), 2),
        "evo/hero in winner deck": round(share_evo(won[wc].to_numpy()), 3),
        "share of wins vs the most common opponent deck": round(won["ld"].value_counts(normalize=True).iloc[0], 2),
    })
print(pd.DataFrame(rows).to_string(index=False))
print("\nFor comparison, over all matches: average card level",
      round(pd.concat([df['wlv'], df['llv']]).mean(), 2),
      "| decks with evo/hero:", round(share_evo(df[wc].to_numpy()), 3))

                                                           deck  avg card level (winner)  avg card level (opponent)  evo/hero in winner deck  share of wins vs the most common opponent deck
knight | archers | goblins | giant | musketeer | mini p.e.k....                     4.05                       3.56                      0.0                                            0.33
knight | archers | goblins | giant | musketeer | mini p.e.k....                     3.53                       3.30                      0.0                                            0.61
knight | archers | goblins | giant | minions | musketeer | m...                     3.34                       3.35                      0.0                                            0.54



For comparison, over all matches: average card level 13.36 | decks with evo/hero: 0.909


The "best" decks have an **average card level of about 4** (the overall average is 13.4), contain **no evolution or hero cards**, and often beat the same opponent deck (33-61% of their wins come against a single opponent deck). These are beginner matches, with a completely different dynamic from the rest of the ladder.

How many matches are like that, and what happens if we remove them?

In [6]:
df["max_lvl"] = np.maximum(df["wlv"], df["llv"])
beginner = df["max_lvl"] < 8
print(f"Beginner matches (both decks have average level < 8): {beginner.sum():,} ({beginner.mean():.1%})\n")

summarise(df, "ALL ladder matches")
summarise(df[~beginner], "WITHOUT beginner matches")

Beginner matches (both decks have average level < 8): 386,024 (3.9%)



ALL ladder matches
  matches 9,960,921 | decks with >=300 games: 2,277
  win-rate quantiles (1/10/50/90/99%): [0.071, 0.148, 0.269, 0.634, 0.861] | max 0.932 | min 0.020
  match-weighted mean win rate: 0.329  (should be close to 0.5)


WITHOUT beginner matches
  matches 9,574,897 | decks with >=300 games: 2,029
  win-rate quantiles (1/10/50/90/99%): [0.085, 0.157, 0.278, 0.632, 0.684] | max 0.750 | min 0.019
  match-weighted mean win rate: 0.333  (should be close to 0.5)


Removing just **3.9% of the matches** brings the maximum deck win rate from 93% down to 75%, and the "robust decks" ranking of the original notebook disappears.

## 4. Cause B: a small group of very active accounts that almost always lose
Even after removing beginners, popular decks have a match-weighted average win rate of about 33%. In a fair sample it must be close to 50%, because every match has exactly one winner and one loser. So some players are losing a lot. Let's look at win rate by how often each player appears in the data.

In [7]:
pw, pl = df["winner_id"].value_counts(), df["loser_id"].value_counts()
P = pd.concat([pw.rename("w"), pl.rename("l")], axis=1).fillna(0)
P["n"] = P["w"] + P["l"]
P["appearances"] = pd.cut(P["n"], [0, 1, 2, 4, 9, 19, 40, 1000],
                          labels=["1", "2", "3-4", "5-9", "10-19", "20-40", ">40"])
g = P.groupby("appearances", observed=True).agg(players=("n", "size"), games=("n", "sum"), wins=("w", "sum"))
g["win_rate"] = (g["wins"] / g["games"]).round(3)
g[["players", "games", "win_rate"]]

,players,games,win_rate
appearances,,,
1,3777429,3777429.0,0.464
2,1073370,2146740.0,0.468
3-4,618898,2066336.0,0.492
5-9,448411,2962932.0,0.553
10-19,348792,4754031.0,0.548
20-40,147926,3740763.0,0.495
>40,6119,473611.0,0.202


The 6,119 players who appear **more than 40 times** win only **20%** of their games. No other group looks like that. Most likely these are automated or heavily farmed accounts, and they play the most popular decks, which explains why those decks look weak (10-21%) while rarer decks look strong.

The original notebook limited the share of a *single* player to 10% of a deck's games. That check cannot detect this problem, because each account contributes little and it is the *group* that matters.

## 5. Other things worth knowing

In [8]:
daily = df["time"].dt.date.value_counts().sort_index()
print("Matches per day (first 3 and last 3 days):")
print(pd.concat([daily.head(3), daily.tail(3)]).to_string())

Matches per day (first 3 and last 3 days):
time
2026-05-22      80430
2026-05-23     376462
2026-05-24     504472
2026-06-02    1224544
2026-06-03    1420879
2026-06-04     122481


The volume grows from about 80,000 matches on the first day to 1.4 million by 3 June, so **the collection is not stationary**: the sample at the end of the period is very different from the one at the start. Any model that mixes days should account for it.

## 6. Conclusions and what to change in the analysis

| Problem | Evidence | Fix |
|---|---|---|
| Beginner matches inflate some decks to 85-93% | Average level about 4, no evo/hero, same opponent deck | Keep only matches where both decks have an average level of at least 8 (or higher) |
| A group of very active accounts at 20% win rate distorts popular decks | 6,119 players with more than 40 appearances | Exclude them, or cap each player's number of matches |
| Collection volume grows during the period | 80k to 1.4M matches per day | Weight or stratify by day |
| Sampling method of the dataset is undocumented | The Kaggle page does not describe it | State it as a limitation in the README |

**Sanity check to pass before trusting any deck ranking:** the match-weighted mean win rate of the decks must return close to 50%. After removing beginners it is still 33%, so the second fix has to be applied and verified as well.

This audit covers one file only. Re-run it on a few more files before generalising.